<a href="https://colab.research.google.com/github/juliaR0/SF_10_new/blob/main/notebooks/S_Mission2_Step4_Baseline1_0_112726.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mission 2, Step 4 — TL1.0 baseline, clinical and natural test modes

*Code drafted with Claude (Anthropic); run, checked and adapted by me. Prompts are in my AI log.*

**What this is:** the first test of my preregistration (PREREGISTRATION.md v1.0), on toy language 1.0 and the 2.1 twins. Per the preregistration's testing schedule, 1.0 can test **B1 (partly), B3, B4, B5, B6 (partly), B7**. Every test runs in **both modes**: **clinical mode** (a session in one language: the target language stays on, no `--cs`; the preregistered main mode, so its rows get PASS/FAIL) and **natural mode** (a familiar bilingual setting: both languages on, `--cs`; reported separately, so its rows are EXPLORATORY with a note saying what the rule would give). The twins are bilingual in both modes; only the test setting differs. Every row in the results file has a `mode` column.

**Every prediction in the preregistration (A1–A3, B1–B9, C1–C5) gets a row** in one results file, `analysis/prereg_results_TL1.0.csv`, built up cell by cell:
- **PASS / FAIL / INCONCLUSIVE**: a confirmatory prediction the TL1.0 schedule allows, judged by its preregistered rule.
- **UNTESTABLE**: blocked by the preregistered testability rule (healthy strict < 50% in that language).
- **NOT_TESTABLE_ON_TL1.0**: TL1.0 can't test it at all; the row says why.
- **EXPLORATORY**: computed and reported, but not in the TL1.0 schedule (or exploratory in the preregistration), so it can't pass or fail.

The CSV has only model results (no Chen data), so it's safe to push to GitHub.

**No new training.** Only evaluations of saved weights, plus new lesions (grammar levels 0.65/0.75 and the meaning-path control lesion).

**Runtime:** use the TPU runtime (24 CPUs → 22 workers). About 276 short runs, ~20–30 min. Run Section 1 → Section 2 (all runs) → Sections 3–5 with `MODE = 'clinical'` → change `MODE` to `'natural'` in Section 3 and run Sections 3–5 again → Section 6.

# 1. Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/dual_path

Mounted at /content/drive
/content/drive/MyDrive/dual_path


In [3]:
## 0.2.1 remove extra things from "requirements.txt"
!sed 's/==.*//' requirements.txt > requirements_unpinned.txt

## 0.2.2 install things in new version "requirements_unpinned.txt"
!pip install -q -r requirements_unpinned.txt

In [4]:
import sys, os, glob, math, copy, pickle, shutil, subprocess
import concurrent.futures as cf
import pandas as pd, numpy as np
from scipy import stats
import lz4.frame, torch
sys.path.insert(0, 'src')
pd.set_option('display.max_rows', 200, 'display.max_colwidth', 100)

PATIENTS = ['P14', 'P77']
N_TWINS = 3
FINAL_EPOCH = 15
OPTIONS = ['EN', 'ES', 'MIXED', 'SHAM']
LEVELS = [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8]     # preregistered
TARGET_RATIO = 0.60                                                   # preregistered
TESTABLE_MIN = 50                                                     # preregistered testability rule (healthy strict %)
# Dominant language (preregistered): higher pre-stroke use; P14 English use 7% -> es; P77 English use 56% -> en
ENG_USE = {'P14': 0.07, 'P77': 0.56}
DOMINANT = {p: ('en' if u > 0.5 else 'es') for p, u in ENG_USE.items()}
print('dominant language:', DOMINANT)

def lang_of(msg): return 'es' if 'TARGET-LANG=es' in str(msg) else 'en'
def structure(msg):
    msg = str(msg)
    if 'RECIPIENT=' in msg: return '3 ditransitive'
    if 'PATIENT=' in msg:   return '2 transitive'
    if 'ATTR=' in msg:      return 'copular'
    return '1 intransitive'

LEX = pd.read_csv('data/code-switching/lexicon.csv')
ES_ONLY = set(LEX.morpheme_es.dropna()) - set(LEX.morpheme_en.dropna())
EN_ONLY = set(LEX.morpheme_en.dropna()) - set(LEX.morpheme_es.dropna())

def load_test(path, epoch=0):
    d = pd.read_csv(path)
    d = d[d.epoch == epoch].copy()
    d['m'] = d['meaning'].astype(str).str.strip()
    d['tlang'] = d.message.apply(lang_of)
    d['struct'] = d.message.apply(structure)
    def lang_ok(r):
        words = set(str(r.produced_sentence).split())
        return not (words & (ES_ONLY if r.tlang == 'en' else EN_ONLY))
    d['lang_ok'] = d.apply(lang_ok, axis=1)
    d['strict'] = (d.m == 'True') & d.lang_ok
    return d

def pct(mask): return round(100 * mask.mean(), 1)

# --- error categories (same as prototype 2.2, fixed version) ---
WORD = {}
for r in LEX.itertuples():
    pos = str(r.pos).strip()
    ta = ':'.join(str(x) for x in [r.tense, r.aspect] if isinstance(x, str))
    for w in [r.morpheme_es, r.morpheme_en]:
        if isinstance(w, str): WORD.setdefault(w.strip(), (pos, ta, str(r.concept)))
for w in set(LEX[LEX.pos == 'aux'].morpheme_es.dropna()) | set(LEX[LEX.pos == 'aux'].morpheme_en.dropna()):
    WORD[w.strip()] = ('aux', 'present', 'AUX')
CONTENT = {'noun', 'verb', 'participle', 'adj', 'prep', 'pron'}
def analyse(sentence):
    toks = [t for t in str(sentence).split() if t != '.']
    info = [WORD.get(t, ('?', '', t)) for t in toks]
    skeleton = tuple('verb' if p == 'participle' else p for p, _, _ in info if p in CONTENT)
    tense = tuple((p, ta) for (p, ta, _) in info if p in ('verb', 'participle', 'aux'))
    dets = tuple(t for t, (p, _, _) in zip(toks, info) if p == 'det')
    concepts = tuple(c for (p, _, c) in info if p in ('noun', 'verb', 'participle', 'adj'))
    return skeleton, tense, dets, concepts
def category(r):
    if r.strict: return 'correct'
    ps, pt, pd_, pc = analyse(r.produced_sentence)
    ts, tt, td, tc = analyse(r.target_sentence)
    if not r.lang_ok and (r.m == 'True' or (ps == ts and pc == tc)): return 'switch'
    repeats = len(set(pc)) < len(pc) and len(set(tc)) == len(tc)
    if ps != ts or repeats: return 'structure'
    if pt != tt: return 'tense/aux'
    if pc != tc: return 'wrong word'
    if pd_ != td: return 'article'
    return 'agreement'
CATS = ['correct', 'switch', 'structure', 'tense/aux', 'wrong word', 'article', 'agreement']
GRAMMAR = ['structure', 'tense/aux', 'article', 'agreement']          # preregistered B3 grouping
def categorise(d): d = d.copy(); d['cat'] = d.apply(category, axis=1); return d

def ci95(x):
    x = pd.Series(x, dtype=float).dropna(); n = len(x)
    if n < 2: return pd.Series({'mean': x.mean(), 'lo': np.nan, 'hi': np.nan, 'n': n})
    m, sd = x.mean(), x.std(ddof=1); h = stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)
    return pd.Series({'mean': round(m, 1), 'lo': round(m - h, 1), 'hi': round(m + h, 1), 'n': n})

# --- weights, lesions, runs (same as 2.1 / 2.2) ---
def load_weights(path):
    with lz4.frame.open(path, 'rb') as f: return pickle.load(f)
def lesion(layers, layer_name, fraction, seed=0):
    d = copy.deepcopy(layers); W = d[layer_name].in_weights
    g = torch.Generator().manual_seed(seed)
    d[layer_name].in_weights = W.masked_fill(torch.rand(W.shape, generator=g) < fraction, 0.0)
    return d
def save_layers(layers, folder, epoch):
    for path in [f'{folder}/1/weights', f'{folder}/weights']:
        os.makedirs(path, exist_ok=True)
        with lz4.frame.open(f'{path}/w{epoch}.lz4', 'wb') as f: pickle.dump(layers, f, protocol=-1)
TINY_TRAIN = True   # SPEED FIX #1: evaluations only need the epoch-0 test, so the throwaway training epoch gets a 5-sentence training file
def build_input(folder, src_run, src_sim, tiny=None):
    tiny = TINY_TRAIN if tiny is None else tiny
    os.makedirs(f'{folder}/input', exist_ok=True); os.makedirs(f'{folder}/1', exist_ok=True)
    for f in os.listdir(f'{src_run}/input'):
        p = f'{src_run}/input/{f}'
        if os.path.isfile(p): shutil.copyfile(p, f'{folder}/input/{f}')
    shutil.copyfile(f'{src_run}/{src_sim}/test.in', f'{folder}/1/test.in')
    lines = open(f'{src_run}/{src_sim}/training.in').read().splitlines(True)
    open(f'{folder}/1/training.in', 'w').writelines(lines[:5] if tiny else lines)
    return folder

WORKERS = max(1, os.cpu_count() - 2)
def run_many(jobs):
    # a run counts as done only if its test.csv exists (an interrupted run leaves a folder without one)
    done = lambda r: os.path.exists(f'simulations/{r}/1/test.csv')
    for a, r in jobs:
        if os.path.exists(f'simulations/{r}') and not done(r): shutil.rmtree(f'simulations/{r}')   # clear half-finished runs
    todo = [(a, r) for a, r in jobs if not done(r)]
    print(len(jobs) - len(todo), 'already done;', len(todo), 'to run on', WORKERS, 'workers')
    env = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
    def one(job):
        a, r = job
        p = subprocess.run(f'python src/start_dual_path.py {a} --resdir {r}', shell=True, capture_output=True, text=True, env=env)
        return r, p.returncode, p.stderr[-400:]
    with cf.ThreadPoolExecutor(WORKERS) as ex:
        for r, c, err in ex.map(one, todo):
            print('ok    ' if c == 0 else 'FAILED', r, '' if c == 0 else err)

# ===== Preregistration registry and results file =====
TL_VERSION = '1.0'
PROTOTYPE = 'Bravo'
RESULTS_PATH = f'analysis/prereg_results_TL{TL_VERSION}.csv'
PREREG_IDS = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'C1', 'C2', 'C3', 'C4', 'C5']
PREREG_TYPE = {i: 'confirmatory' for i in PREREG_IDS} | {'B8': 'exploratory', 'B9': 'exploratory', 'C5': 'exploratory'}
SCHEDULE_TL1 = {'B1', 'B3', 'B4', 'B5', 'B6', 'B7'}          # preregistered TL1.0 testing schedule
NOT_ON_TL1 = {'A1': 'needs the test-battery sections, which only exist in TL2.0',
              'B2': 'TL1.0 has no passives (no non-canonical sentences)',
              'B9': 'TL1.0 verbs are whole words, so endings cannot be dropped or swapped',
              'C3': 'TL1.0 was not designed with shared vs language-specific structures',
              'C4': 'no maintenance-therapy runs exist on TL1.0'}
# What I predicted BEFORE running (copied from PREREGISTRATION.md v1.0), plus a number to plot against
# id: (prediction, pass rule, metric that `estimate` holds, threshold, comparator)
SPEC = {
 'A1': ('Healthy twins near ceiling in dominant language on every battery section', '>= 85% on every section', 'healthy strict % (dominant language)', 85, '>='),
 'A2': ('Weaker language scales with exposure', 'more-exposed twin scores higher in L2', 'L2 strict %: more-exposed minus less-exposed', 0, '>'),
 'A3': ('Healthy twins rarely switch (single-language mode)', '<= 5% switches', 'healthy switch %', 5, '<='),
 'B1': ('More arguments = more errors', '1 > 2 > 3-arg and interval of (1-arg minus 3-arg) above 0', 'strict %: 1-arg minus 3-arg', 0, '>'),
 'B2': ('Passives harder than actives', 'agrammatism ratio < 1, interval below 1', 'passive / active accuracy', 1, '<'),
 'B3': ('Grammar damage hits grammar, not words', 'grammar-error rise >= 10 points more than wrong-word rise', 'grammar rise minus wrong-word rise (points)', 10, '>='),
 'B4': ('Meaning damage hits words (double dissociation)', 'wrong-word rise > grammar rise', 'wrong-word rise minus grammar rise (points)', 0, '>'),
 'B5': ('More errors in non-dominant language', 'non-dominant > dominant', 'error %: non-dominant minus dominant', 0, '>'),
 'B6': ('Same error-type mix in both languages', 'same top-2 error types', 'top-2 error types shared (0-2)', 2, '=='),
 'B7': ('Switches go mostly into dominant language', '> 50% into dominant', '% of switches into dominant language', 50, '>'),
 'B8': ('Tense errors more common than agreement errors', 'exploratory', 'tense/aux % minus agreement %', 0, '>'),
 'B9': ('English drops endings, Spanish swaps them', 'exploratory', 'omission share minus substitution share', None, ''),
 'C1': ('Therapy helps trained language beyond sham', 'interval above 0', 'sham-adjusted strict gain (points)', 0, '>'),
 'C2': ('Hidden transfer: untrained language structure errors fall beyond sham', 'interval below 0', 'sham-adjusted change in structure errors (points)', 0, '<'),
 'C3': ('Transfer larger for shared structures', 'interval of (shared minus specific) above 0', 'transfer: shared minus specific', 0, '>'),
 'C4': ('Maintenance reduces forgetting', 'interval of difference above 0', 'forgetting without minus with maintenance', 0, '>'),
 'C5': ('Training complex sentences helps simpler ones (CATE)', 'exploratory', 'sham-adjusted strict gain on simpler structures', 0, '>'),
}
# Two test modes (J's names): CLINICAL mode = a session in one language, the target language stays on (preregistered main mode, 'single-language')
# NATURAL mode = a familiar bilingual setting, both languages on (--cs); reported separately, so its rows are EXPLORATORY
MODES = {'clinical': ('', '_sl'), 'natural': ('--cs', '_bl')}
MODE = 'clinical'          # Section 3 sets this; run Sections 3-5 once per mode
RESULTS = []
def record(pid_, patient='all', lang='all', estimate=None, lo=None, hi=None, n=None, value='', status='', note='', cell=''):
    if not status:
        status = 'EXPLORATORY' if (PREREG_TYPE[pid_] == 'exploratory' or pid_ not in SCHEDULE_TL1) else 'INCONCLUSIVE'
    mode = 'both' if status == 'NOT_TESTABLE_ON_TL1.0' and cell == '1 Setup' else MODE
    if mode == 'natural' and status in ('PASS', 'FAIL', 'INCONCLUSIVE', 'UNTESTABLE'):
        note = (note + '; ' if note else '') + f'natural mode (--cs), reported separately; the rule gives {status}'
        status = 'EXPLORATORY'
    pred, rule, metric, thr, cmp_ = SPEC[pid_]
    f = lambda v: None if v is None or (isinstance(v, float) and np.isnan(v)) else round(float(v), 2)
    RESULTS.append({'tl_version': TL_VERSION, 'prototype': PROTOTYPE, 'mode': mode, 'id': pid_, 'type': PREREG_TYPE[pid_],
                    'in_TL1_schedule': pid_ in SCHEDULE_TL1, 'patient': patient, 'lang': lang,
                    'expected': pred, 'pass_rule': rule, 'metric': metric, 'threshold': thr, 'comparator': cmp_,
                    'estimate': f(estimate), 'ci_lo': f(lo), 'ci_hi': f(hi), 'n_twins': n,
                    'status': status, 'value_text': value, 'note': note, 'source_cell': cell,
                    'notebook': 'S_Mission2_Step4_Baseline_112726', 'run_time': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')})
    pd.DataFrame(RESULTS).to_csv(RESULTS_PATH, index=False)          # saved after every row
for i, why in NOT_ON_TL1.items():
    record(i, status='NOT_TESTABLE_ON_TL1.0', note=why, cell='1 Setup')
print(len(RESULTS), 'rows so far ->', RESULTS_PATH)

def base_args(pid, k):
    w = load_weights(f'simulations/p21_{pid}_twins/{k}/weights/w{FINAL_EPOCH}.lz4')
    # NOTE: no --cs here; Section 2 adds --cs only for the bilingual-mode runs
    return w, f"--norandomization --sim 1 --hidden {w['hidden'].size} --compress {w['compress'].size} --fw 15 --lang es en"

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


dominant language: {'P14': 'es', 'P77': 'en'}
5 rows so far -> analysis/prereg_results_TL1.0.csv


# 1b. Speed-fix check (run once before Section 2)

Speed fix #1 gives each evaluation a 5-sentence training file, so the throwaway training epoch takes seconds instead of minutes. The epoch-0 test should be **identical** to a full-training-file run. This cell runs one twin both ways and compares every produced sentence. **It must print `identical: True`.** If it prints `False`, set `TINY_TRAIN = False` in Setup and rerun Setup before Section 2.

In [5]:
import time
pid, k = 'P14', 1
w, args = base_args(pid, k)
run = f'simulations/p21_{pid}_twins'
folder = 'simulations/m2s4_check_w'
if not os.path.exists(f'{folder}/weights/w{FINAL_EPOCH}.lz4'): save_layers(w, folder, FINAL_EPOCH)
res = {}
for tiny in [True, False]:
    tag = f'm2s4_check_{"tiny" if tiny else "full"}'
    inp = build_input(f'simulations/{tag}_in', run, k, tiny=tiny)
    t0 = time.time()
    run_many([(f'{args} --epochs 1 --sw {folder} --swe {FINAL_EPOCH} --input {inp}', f'{tag}_sl')])
    print(tag, 'took', round((time.time() - t0) / 60, 1), 'min')
    d = pd.read_csv(f'simulations/{tag}_sl/1/test.csv'); res[tiny] = d[d.epoch == 0].sort_values('message').produced_sentence.tolist()
print('identical:', res[True] == res[False], '| sentences compared:', len(res[True]))

1 already done; 0 to run on 22 workers
m2s4_check_tiny took 0.0 min
1 already done; 0 to run on 22 workers
m2s4_check_full took 0.0 min
identical: True | sentences compared: 500


# 2. All runs (single-language mode)

Three kinds of evaluation, each **one epoch, reading epoch-0 rows** (the 2.1 loading route):
- **Grammar lesion** (hidden → pred_compress) at every preregistered level, including new 0.65 and 0.75. Level 0.0 = healthy.
- **Meaning lesion** (hidden → pred_role) at every level: the control for the double dissociation (B4).
- **Therapy weights from 2.1, re-tested** (exploratory C2).

Each evaluation runs **twice**: **clinical mode** (no `--cs`, run names end `_sl`; the preregistered main mode) and **natural mode** (`--cs`, names end `_bl`; reported separately). The weights are the same; only the test condition differs. That's ~276 short runs, ~20–30 min on the TPU.

`# RULE` run names end in `_sl` / `_bl` so the modes never mix with each other or with the old 2.1 runs.

⚠️ **First run check:** after the runs, Section 3 prints the vocabulary size of one single-language run vs a `--cs` run. They must match; without `--cs` the code looks for the default lexicon in another folder, and our `--input` should override it.

In [6]:
jobs = []
for pid in PATIENTS:
    run = f'simulations/p21_{pid}_twins'
    for k in range(1, N_TWINS + 1):
        w, args = base_args(pid, k)
        for site, layer in [('g', 'pred_compress'), ('m', 'pred_role')]:
            for frac in LEVELS:
                if site == 'm' and frac == 0.0: continue           # 0.0 is the same for both sites
                tag = f'm2s4_{pid}_t{k}_{site}{int(round(frac*100)):02d}'
                folder = f'simulations/{tag}_w'
                if not os.path.exists(f'{folder}/weights/w{FINAL_EPOCH}.lz4'):
                    save_layers(lesion(w, layer, frac, seed=k), folder, FINAL_EPOCH)
                inp = build_input(f'simulations/{tag}_in', run, k)
                for flag, suf in MODES.values():
                    jobs.append((f'{args} {flag} --epochs 1 --sw {folder} --swe {FINAL_EPOCH} --input {inp}', f'{tag}{suf}'))
        # 2.1 therapy weights, re-tested in single-language mode (EXPLORATORY C2)
        for o in OPTIONS:
            src = f'simulations/p21_{pid}_t{k}_th{o}_run'
            ws = sorted(glob.glob(f'{src}/**/weights/w*.lz4', recursive=True), key=lambda p: int(p.split('/w')[-1].split('.')[0]))
            if not ws: print('missing therapy weights:', src); continue
            last = ws[-1]; ep = int(last.split('/w')[-1].split('.')[0])
            tag = f'm2s4_{pid}_t{k}_th{o}'
            folder = f'simulations/{tag}_w'
            if not os.path.exists(f'{folder}/weights/w{ep}.lz4'):
                save_layers(load_weights(last), folder, ep)
            inp = build_input(f'simulations/{tag}_in', run, k)
            for flag, suf in MODES.values():
                jobs.append((f'{args} {flag} --epochs 1 --sw {folder} --swe {ep} --input {inp}', f'{tag}{suf}'))
print(len(jobs), 'jobs')
run_many(jobs)

276 jobs
69 already done; 207 to run on 22 workers
ok     m2s4_P14_t2_m30_bl 
ok     m2s4_P14_t2_m40_sl 
ok     m2s4_P14_t2_m40_bl 
ok     m2s4_P14_t2_m50_sl 
ok     m2s4_P14_t2_m50_bl 
ok     m2s4_P14_t2_m60_sl 
ok     m2s4_P14_t2_m60_bl 
ok     m2s4_P14_t2_m65_sl 
ok     m2s4_P14_t2_m65_bl 
ok     m2s4_P14_t2_m70_sl 
ok     m2s4_P14_t2_m70_bl 
ok     m2s4_P14_t2_m75_sl 
ok     m2s4_P14_t2_m75_bl 
ok     m2s4_P14_t2_m80_sl 
ok     m2s4_P14_t2_m80_bl 
ok     m2s4_P14_t2_thEN_sl 
ok     m2s4_P14_t2_thEN_bl 
ok     m2s4_P14_t2_thES_sl 
ok     m2s4_P14_t2_thES_bl 
ok     m2s4_P14_t2_thMIXED_sl 
ok     m2s4_P14_t2_thMIXED_bl 
ok     m2s4_P14_t2_thSHAM_sl 
ok     m2s4_P14_t2_thSHAM_bl 
ok     m2s4_P14_t3_g00_sl 
ok     m2s4_P14_t3_g00_bl 
ok     m2s4_P14_t3_g20_sl 
ok     m2s4_P14_t3_g20_bl 
ok     m2s4_P14_t3_g30_sl 
ok     m2s4_P14_t3_g30_bl 
ok     m2s4_P14_t3_g40_sl 
ok     m2s4_P14_t3_g40_bl 
ok     m2s4_P14_t3_g50_sl 
ok     m2s4_P14_t3_g50_bl 
ok     m2s4_P14_t3_g60_sl 
ok     m2s4_P

# 3. Checks first

In [12]:
MODE = 'natural'      # <- run Sections 3-5 with 'clinical' first, then change to 'natural' and run them again
SUF = MODES[MODE][1]
print('analysing mode:', MODE)
# (a) vocabulary check: single-language vs --cs run must produce the same word set
a = pd.read_csv('simulations/m2s4_P77_t1_g00_sl/1/test.csv')
b = pd.read_csv(f'simulations/p21_P77_twins/1/test.csv')
print('target sentences identical:', set(a.target_sentence) == set(b.target_sentence))
# (b) load everything
def path(pid, k, site, frac): return f'simulations/m2s4_{pid}_t{k}_{site}{int(round(frac*100)):02d}{SUF}/1/test.csv'
rows = []
for pid in PATIENTS:
    for k in range(1, N_TWINS + 1):
        for site in ['g', 'm']:
            for frac in LEVELS:
                p = path(pid, k, 'g' if frac == 0.0 else site, frac)
                if not os.path.exists(p): print('missing', p); continue
                x = categorise(load_test(p)); x['patient'], x['twin'], x['site'], x['damage'] = pid, k, site, frac
                rows.append(x)
ALL = pd.concat(rows)
ALL.to_csv(f'analysis/m2s4_all_sentences_TL1.0_{MODE}.csv', index=False)
# (c) healthy single-language accuracy + testability rule (preregistered: healthy strict >= 50%)
H = ALL[(ALL.damage == 0.0) & (ALL.site == 'g')]
healthy = H.groupby(['patient', 'twin', 'tlang']).strict.mean().mul(100).round(1).unstack()
print('healthy strict % (single-language):'); display(healthy)
if MODE == 'clinical':   # preregistered: testability is decided in single-language (clinical) mode
    TESTABLE = {(p, l): healthy.loc[p, l].min() >= TESTABLE_MIN for p in PATIENTS for l in ['en', 'es']}
print('testable languages:', TESTABLE)
display(healthy)
# A1: not testable on TL1.0 (recorded in Setup); add the overall healthy dominant-language score as information only
for pid in PATIENTS:
    if MODE != 'clinical': break
    v = healthy.loc[pid, DOMINANT[pid]]
    record('A1', pid, DOMINANT[pid], estimate=v.mean(), lo=v.min(), hi=v.max(), n=len(v),
           value=f"overall healthy strict per twin {v.round(1).tolist()} (min-max shown as lo-hi; no battery sections)",
           status='NOT_TESTABLE_ON_TL1.0', note='information only', cell='3 Checks')
# A2 (exploratory on TL1.0): the twin with more English exposure scores higher in English
p_hi, p_lo = max(ENG_USE, key=ENG_USE.get), min(ENG_USE, key=ENG_USE.get)
a2 = (healthy.loc[p_hi, 'en'].mean(), healthy.loc[p_lo, 'en'].mean())
record('A2', 'both', 'en', estimate=a2[0] - a2[1], n=N_TWINS, value=f"{p_hi} en {a2[0]:.1f} vs {p_lo} en {a2[1]:.1f}",
       note='would pass' if a2[0] > a2[1] else 'would fail', cell='3 Checks')
# A3 (exploratory on TL1.0): healthy switch % in single-language mode
sw = H.assign(sw=H.cat == 'switch').groupby(['patient', 'tlang']).sw.mean().mul(100).round(1)
for (pid, lang), v in sw.items():
    per_twin = H[(H.patient == pid) & (H.tlang == lang)].assign(sw=lambda d: d.cat == 'switch').groupby('twin').sw.mean().mul(100)
    c = ci95(per_twin)
    record('A3', pid, lang, estimate=c['mean'], lo=c['lo'], hi=c['hi'], n=c['n'], value=f"{v}% switches",
           note='would pass (<= 5%)' if v <= 5 else 'would fail (> 5%)', cell='3 Checks')
display(sw)

analysing mode: natural
target sentences identical: True
healthy strict % (single-language):


tlang           en    es
patient twin            
P14     1      8.0  96.7
        2      6.0  98.2
        3      4.0  98.0
P77     1     33.9  86.8
        2     66.8  80.9
        3     83.9  79.5

testable languages: {('P14', 'en'): np.False_, ('P14', 'es'): np.True_, ('P77', 'en'): np.True_, ('P77', 'es'): np.True_}


tlang           en    es
patient twin            
P14     1      8.0  96.7
        2      6.0  98.2
        3      4.0  98.0
P77     1     33.9  86.8
        2     66.8  80.9
        3     83.9  79.5

patient  tlang
P14      en        4.0
         es        0.4
P77      en       30.0
         es        3.8
Name: sw, dtype: float64

# 4. Damage calibration (preregistered) and the predictions

`# RULE` (preregistered) each twin's damage = the level whose overall strict accuracy is closest to 60% of its healthy accuracy, scored in single-language mode; separately for the grammar and the meaning lesion.

In [13]:
acc = ALL.groupby(['patient', 'twin', 'site', 'damage']).strict.mean().mul(100).rename('acc').reset_index()
h = acc[acc.damage == 0.0].groupby(['patient', 'twin']).acc.first()
acc['ratio'] = acc.apply(lambda r: r.acc / h[(r.patient, r.twin)], axis=1).round(2)
display(acc.pivot_table(index=['patient', 'twin', 'site'], columns='damage', values='ratio'))
if MODE == 'clinical':   # preregistered: damage is calibrated in single-language (clinical) mode; natural mode reuses it
    CHOSEN = {}
    for (pid, k, site), g in acc[acc.damage > 0].groupby(['patient', 'twin', 'site']):
        CHOSEN[(pid, k, site)] = g.loc[(g.ratio - TARGET_RATIO).abs().idxmin(), 'damage']
print('chosen damage:', CHOSEN)
def at_chosen(site):
    parts = []
    for (pid, k, s), f in CHOSEN.items():
        if s != site: continue
        parts.append(ALL[(ALL.patient == pid) & (ALL.twin == k) & (ALL.damage == f) & ((ALL.site == site) | (f == 0.0))])
    return pd.concat(parts)
G, Mn = at_chosen('g'), at_chosen('m')

damage             0.00  0.20  0.30  0.40  0.50  0.60  0.65  0.70  0.75  0.80
patient twin site                                                            
P14     1    g      1.0  1.01  1.01  0.99  0.94  0.80  0.77  0.74  0.67  0.41
             m      1.0  0.90  0.93  0.90  0.90  0.86  0.85  0.65  0.46  0.65
        2    g      1.0  0.95  0.91  0.89  0.75  0.59  0.58  0.56  0.50  0.12
             m      1.0  0.92  0.84  0.78  0.55  0.82  0.67  0.70  0.81  0.80
        3    g      1.0  0.96  0.95  0.91  0.70  0.47  0.32  0.31  0.29  0.31
             m      1.0  0.85  0.78  0.79  0.65  0.70  0.49  0.47  0.38  0.37
P77     1    g      1.0  0.88  0.85  0.70  0.66  0.51  0.36  0.37  0.21  0.13
             m      1.0  0.68  0.68  0.65  0.52  0.38  0.25  0.45  0.22  0.40
        2    g      1.0  0.97  1.03  0.93  0.86  0.70  0.56  0.38  0.14  0.31
             m      1.0  0.81  0.68  0.42  0.06  0.73  0.55  0.27  0.51  0.42
        3    g      1.0  0.98  0.95  0.94  0.83  0.65  0.65  0.58  0.50  0.42
             m      1.0  0.87  0.66  0.57  0.61  0.65  0.55  0.15  0.29  0.28

chosen damage: {('P14', np.int64(1), 'g'): np.float64(0.75), ('P14', np.int64(1), 'm'): np.float64(0.8), ('P14', np.int64(2), 'g'): np.float64(0.75), ('P14', np.int64(2), 'm'): np.float64(0.5), ('P14', np.int64(3), 'g'): np.float64(0.6), ('P14', np.int64(3), 'm'): np.float64(0.65), ('P77', np.int64(1), 'g'): np.float64(0.7), ('P77', np.int64(1), 'm'): np.float64(0.5), ('P77', np.int64(2), 'g'): np.float64(0.65), ('P77', np.int64(2), 'm'): np.float64(0.65), ('P77', np.int64(3), 'g'): np.float64(0.75), ('P77', np.int64(3), 'm'): np.float64(0.4)}


In [14]:
def testable(pid, lang): return TESTABLE[(pid, lang)]
def rec(pid, lang, rid, value, verdict, c=None, est=None, n=None):   # B-rows: verdicts are upper-case statuses
    if c is not None: record(rid, pid, lang, estimate=c['mean'], lo=c['lo'], hi=c['hi'], n=c['n'], value=value, status=verdict.upper(), cell='4 Verdicts')
    else: record(rid, pid, lang, estimate=est, n=n, value=value, status=verdict.upper(), cell='4 Verdicts')

# B1 (partly): 1-arg > 2-arg > 3-arg; interval of (1-arg - 3-arg) above 0
for pid in PATIENTS:
    for lang in ['en', 'es']:
        if not testable(pid, lang): rec(pid, lang, 'B1', '', 'untestable'); continue
        d = G[(G.patient == pid) & (G.tlang == lang) & (G.struct != 'copular')]
        per = d.groupby(['twin', 'struct']).strict.mean().mul(100).unstack()
        means = per.mean().round(1).to_dict()
        diff = ci95(per['1 intransitive'] - per['3 ditransitive'])
        order_ok = means['1 intransitive'] > means['2 transitive'] > means['3 ditransitive']
        verdict = 'pass' if order_ok and diff['lo'] > 0 else ('inconclusive' if order_ok else 'fail')
        rec(pid, lang, 'B1', f"{means}; 1-3 = {diff['mean']} [{diff['lo']}, {diff['hi']}]", verdict, c=diff)

# B3 grammar errors rise >= 10 points more than wrong-word errors (grammar lesion); B4 opposite for meaning lesion
def rises(D, pid, lang):
    out = []
    for k in range(1, N_TWINS + 1):
        hh = H[(H.patient == pid) & (H.twin == k) & (H.tlang == lang)]
        dd = D[(D.patient == pid) & (D.twin == k) & (D.tlang == lang)]
        g_rise = 100 * (dd.cat.isin(GRAMMAR).mean() - hh.cat.isin(GRAMMAR).mean())
        w_rise = 100 * ((dd.cat == 'wrong word').mean() - (hh.cat == 'wrong word').mean())
        out.append((g_rise, w_rise))
    return pd.DataFrame(out, columns=['grammar_rise', 'wordrise'])
for pid in PATIENTS:
    for lang in ['en', 'es']:
        if not testable(pid, lang):
            rec(pid, lang, 'B3', '', 'untestable'); rec(pid, lang, 'B4', '', 'untestable'); continue
        r = rises(G, pid, lang); diff = ci95(r.grammar_rise - r.wordrise)
        rec(pid, lang, 'B3', f"grammar-word = {diff['mean']} [{diff['lo']}, {diff['hi']}]", 'pass' if diff['mean'] >= 10 else 'fail', c=diff)
        r = rises(Mn, pid, lang); diff = ci95(r.wordrise - r.grammar_rise)
        rec(pid, lang, 'B4', f"word-grammar = {diff['mean']} [{diff['lo']}, {diff['hi']}]", 'pass' if diff['mean'] > 0 else 'fail', c=diff)

# B5 more errors in the non-dominant language (grammar lesion), per patient
for pid in PATIENTS:
    dom = DOMINANT[pid]; non = 'en' if dom == 'es' else 'es'
    if not (testable(pid, dom) and testable(pid, non)): rec(pid, 'both', 'B5', '', 'untestable'); continue
    e = G[G.patient == pid].groupby('tlang').strict.mean().rsub(1).mul(100).round(1)
    per = G[G.patient == pid].groupby(['twin', 'tlang']).strict.mean().rsub(1).mul(100).unstack()
    rec(pid, 'both', 'B5', f"errors {dict(e)}", 'pass' if e[non] > e[dom] else 'fail', c=ci95(per[non] - per[dom]))

# B6 (partly) same top-2 error types in both languages
for pid in PATIENTS:
    if not (testable(pid, 'en') and testable(pid, 'es')): rec(pid, 'both', 'B6', '', 'untestable'); continue
    top = {l: list(G[(G.patient == pid) & (G.tlang == l) & (G.cat != 'correct')].cat.value_counts().index[:2]) for l in ['en', 'es']}
    shared = len(set(top['en']) & set(top['es']))
    rec(pid, 'both', 'B6', f"{top}", 'pass' if set(top['en']) == set(top['es']) else 'fail', est=shared, n=N_TWINS)

# B7 switches go mostly into the dominant language (switches happen when the target is the non-dominant language)
for pid in PATIENTS:
    dom = DOMINANT[pid]
    sw = G[(G.patient == pid) & (G.cat == 'switch')]
    if len(sw) == 0: rec(pid, 'both', 'B7', 'no switches', 'inconclusive'); continue
    into_dom = (sw.tlang != dom).mean() * 100
    rec(pid, 'both', 'B7', f"{round(into_dom, 1)}% of {len(sw)} switches into {dom}", 'pass' if into_dom > 50 else 'fail', est=into_dom, n=N_TWINS)

# B8 (exploratory): tense/aux errors vs agreement errors, damaged twins
for pid in PATIENTS:
    for lang in ['en', 'es']:
        d = G[(G.patient == pid) & (G.tlang == lang)]
        per = d.groupby('twin').cat.apply(lambda c: 100 * ((c == 'tense/aux').mean() - (c == 'agreement').mean()))
        c = ci95(per)
        record('B8', pid, lang, estimate=c['mean'], lo=c['lo'], hi=c['hi'], n=c['n'],
               value=f"tense/aux {100*(d.cat=='tense/aux').mean():.1f}% vs agreement {100*(d.cat=='agreement').mean():.1f}%",
               note='' if testable(pid, lang) else 'language untestable; shown for information', cell='4 Verdicts')
pd.DataFrame(RESULTS)

,tl_version,prototype,mode,id,type,in_TL1_schedule,patient,lang,expected,pass_rule,...,estimate,ci_lo,ci_hi,n_twins,status,value_text,note,source_cell,notebook,run_time
0,1.0,Bravo,both,A1,confirmatory,False,all,all,Healthy twins near ceiling in dominant language on every battery section,>= 85% on every section,...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,"needs the test-battery sections, which only exist in TL2.0",1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
1,1.0,Bravo,both,B2,confirmatory,False,all,all,Passives harder than actives,"agrammatism ratio < 1, interval below 1",...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,TL1.0 has no passives (no non-canonical sentences),1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
2,1.0,Bravo,both,B9,exploratory,False,all,all,"English drops endings, Spanish swaps them",exploratory,...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,"TL1.0 verbs are whole words, so endings cannot be dropped or swapped",1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
3,1.0,Bravo,both,C3,confirmatory,False,all,all,Transfer larger for shared structures,interval of (shared minus specific) above 0,...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,TL1.0 was not designed with shared vs language-specific structures,1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
4,1.0,Bravo,both,C4,confirmatory,False,all,all,Maintenance reduces forgetting,interval of difference above 0,...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,no maintenance-therapy runs exist on TL1.0,1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
5,1.0,Bravo,clinical,A1,confirmatory,False,P14,es,Healthy twins near ceiling in dominant language on every battery section,>= 85% on every section,...,98.90,98.2,99.6,3.0,NOT_TESTABLE_ON_TL1.0,"overall healthy strict per twin [99.6, 98.2, 98.9] (min-max shown as lo-hi; no battery sections)",information only,3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
6,1.0,Bravo,clinical,A1,confirmatory,False,P77,en,Healthy twins near ceiling in dominant language on every battery section,>= 85% on every section,...,95.47,94.6,96.1,3.0,NOT_TESTABLE_ON_TL1.0,"overall healthy strict per twin [94.6, 95.7, 96.1] (min-max shown as lo-hi; no battery sections)",information only,3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
7,1.0,Bravo,clinical,A2,confirmatory,False,both,en,Weaker language scales with exposure,more-exposed twin scores higher in L2,...,86.80,NaN,NaN,3.0,EXPLORATORY,P77 en 95.5 vs P14 en 8.7,would pass,3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
8,1.0,Bravo,clinical,A3,confirmatory,False,P14,en,Healthy twins rarely switch (single-language mode),<= 5% switches,...,0.00,0.0,0.0,3.0,EXPLORATORY,0.0% switches,would pass (<= 5%),3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
9,1.0,Bravo,clinical,A3,confirmatory,False,P14,es,Healthy twins rarely switch (single-language mode),<= 5% switches,...,0.00,0.0,0.0,3.0,EXPLORATORY,0.0% switches,would pass (<= 5%),3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27


# 5. EXPLORATORY — C2 on toy language 1.0 (hidden transfer, single-language)

Not in the preregistered 1.0 schedule, so it's **exploratory**: re-testing the 2.1 therapy weights in single-language mode. Question: after therapy in one language, do the **untrained** language's structure errors fall more than after sham? Ditransitives only (the trained structure). "Before" = the 2.1 damaged twin at its 2.1 chosen level, re-tested in single-language mode.

In [15]:
CH21 = {('P14', 1): 0.6, ('P14', 2): 0.6, ('P14', 3): 0.5, ('P77', 1): 0.5, ('P77', 2): 0.6, ('P77', 3): 0.6}
rows = []
for pid in PATIENTS:
    for k in range(1, N_TWINS + 1):
        before_all = categorise(load_test(path(pid, k, 'g', CH21[(pid, k)])))
        for o in OPTIONS:
            p = f'simulations/m2s4_{pid}_t{k}_th{o}{SUF}/1/test.csv'
            if not os.path.exists(p): continue
            after_all = categorise(load_test(p))
            for lang in ['en', 'es']:
              for grp, structs in [('ditransitive', ['3 ditransitive']), ('simpler', ['1 intransitive', '2 transitive'])]:
                b = before_all[(before_all.tlang == lang) & before_all.struct.isin(structs)]
                a = after_all[(after_all.tlang == lang) & after_all.struct.isin(structs)]
                rows.append({'patient': pid, 'twin': k, 'option': o, 'lang': lang, 'group': grp,
                             'structure_change': 100 * ((a.cat == 'structure').mean() - (b.cat == 'structure').mean()),
                             'strict_change': 100 * (a.strict.mean() - b.strict.mean()),
                             'switch_after': 100 * (a.cat == 'switch').mean()})
TC = pd.DataFrame(rows)
TC.to_csv(f'analysis/m2s4_therapy_changes_TL1.0_{MODE}.csv', index=False)
def sham_adj(pid, o, lang, grp, col):
    x = TC[(TC.patient == pid) & (TC.option == o) & (TC.lang == lang) & (TC.group == grp)].set_index('twin')[col]
    s_ = TC[(TC.patient == pid) & (TC.option == 'SHAM') & (TC.lang == lang) & (TC.group == grp)].set_index('twin')[col]
    return ci95(x - s_)
trained = {'EN': 'en', 'ES': 'es'}; untrained = {'EN': 'es', 'ES': 'en'}
for pid in PATIENTS:
    for o in ['EN', 'ES']:
        t, u = trained[o], untrained[o]
        c = sham_adj(pid, o, t, 'ditransitive', 'strict_change')
        record('C1', pid, t, estimate=c['mean'], lo=c['lo'], hi=c['hi'], n=c['n'], cell='5 Therapy',
               value=f"{o} therapy: sham-adj strict gain {c['mean']} [{c['lo']}, {c['hi']}]",
               note=('would pass' if c['lo'] > 0 else 'would not pass') + ('' if TESTABLE[(pid, t)] else '; language untestable'))
        c = sham_adj(pid, o, u, 'ditransitive', 'structure_change')
        sw_after = TC[(TC.patient == pid) & (TC.option == o) & (TC.lang == u) & (TC.group == 'ditransitive')].switch_after.mean()
        record('C2', pid, u, estimate=c['mean'], lo=c['lo'], hi=c['hi'], n=c['n'], cell='5 Therapy',
               value=f"{o} therapy: sham-adj structure-error change {c['mean']} [{c['lo']}, {c['hi']}]; switch after {sw_after:.1f}%",
               note=('would pass' if c['hi'] < 0 else 'would not pass') + ('' if TESTABLE[(pid, u)] else '; language untestable'))
        c = sham_adj(pid, o, t, 'simpler', 'strict_change')
        record('C5', pid, t, estimate=c['mean'], lo=c['lo'], hi=c['hi'], n=c['n'], cell='5 Therapy',
               value=f"{o} therapy: sham-adj strict gain on untrained simpler structures {c['mean']} [{c['lo']}, {c['hi']}]",
               note='' if TESTABLE[(pid, t)] else 'language untestable')
pd.DataFrame(RESULTS).query("id in ['C1', 'C2', 'C5']")

,tl_version,prototype,mode,id,type,in_TL1_schedule,patient,lang,expected,pass_rule,...,estimate,ci_lo,ci_hi,n_twins,status,value_text,note,source_cell,notebook,run_time
34,1.0,Bravo,clinical,C1,confirmatory,False,P14,en,Therapy helps trained language beyond sham,interval above 0,...,0.0,0.0,0.0,3.0,EXPLORATORY,"EN therapy: sham-adj strict gain 0.0 [0.0, 0.0]",would not pass; language untestable,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
35,1.0,Bravo,clinical,C2,confirmatory,False,P14,es,Hidden transfer: untrained language structure errors fall beyond sham,interval below 0,...,7.9,-22.3,38.0,3.0,EXPLORATORY,"EN therapy: sham-adj structure-error change 7.9 [-22.3, 38.0]; switch after 2.8%",would not pass,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
36,1.0,Bravo,clinical,C5,exploratory,False,P14,en,Training complex sentences helps simpler ones (CATE),exploratory,...,1.8,-13.1,16.6,3.0,EXPLORATORY,"EN therapy: sham-adj strict gain on untrained simpler structures 1.8 [-13.1, 16.6]",language untestable,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
37,1.0,Bravo,clinical,C1,confirmatory,False,P14,es,Therapy helps trained language beyond sham,interval above 0,...,46.2,-17.3,109.7,3.0,EXPLORATORY,"ES therapy: sham-adj strict gain 46.2 [-17.3, 109.7]",would not pass,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
38,1.0,Bravo,clinical,C2,confirmatory,False,P14,en,Hidden transfer: untrained language structure errors fall beyond sham,interval below 0,...,-43.2,-101.1,14.8,3.0,EXPLORATORY,"ES therapy: sham-adj structure-error change -43.2 [-101.1, 14.8]; switch after 36.5%",would not pass; language untestable,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
39,1.0,Bravo,clinical,C5,exploratory,False,P14,es,Training complex sentences helps simpler ones (CATE),exploratory,...,-8.3,-19.7,3.0,3.0,EXPLORATORY,"ES therapy: sham-adj strict gain on untrained simpler structures -8.3 [-19.7, 3.0]",,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
40,1.0,Bravo,clinical,C1,confirmatory,False,P77,en,Therapy helps trained language beyond sham,interval above 0,...,28.8,-5.9,63.4,3.0,EXPLORATORY,"EN therapy: sham-adj strict gain 28.8 [-5.9, 63.4]",would not pass,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
41,1.0,Bravo,clinical,C2,confirmatory,False,P77,es,Hidden transfer: untrained language structure errors fall beyond sham,interval below 0,...,6.8,-55.9,69.5,3.0,EXPLORATORY,"EN therapy: sham-adj structure-error change 6.8 [-55.9, 69.5]; switch after 4.8%",would not pass,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
42,1.0,Bravo,clinical,C5,exploratory,False,P77,en,Training complex sentences helps simpler ones (CATE),exploratory,...,-3.1,-10.8,4.6,3.0,EXPLORATORY,"EN therapy: sham-adj strict gain on untrained simpler structures -3.1 [-10.8, 4.6]",,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
43,1.0,Bravo,clinical,C1,confirmatory,False,P77,es,Therapy helps trained language beyond sham,interval above 0,...,50.3,35.3,65.3,3.0,EXPLORATORY,"ES therapy: sham-adj strict gain 50.3 [35.3, 65.3]",would pass,5 Therapy,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27


# 6. Completeness check + final results table

Run this after Sections 3-5 have been run in **both** modes. Every preregistered ID must appear at least once per mode. The file `analysis/prereg_results_TL1.0.csv` is the one to push to GitHub.

Note: the gates (HG → AG → TG) are not applied on the TL1.0 baseline, because HG can't be tested without the battery (A1). The baseline is a "before" measurement, not a gate decision.

In [16]:
R = pd.DataFrame(RESULTS)
for m in ['clinical', 'natural']:
    ids = set(R[(R['mode'] == m) | (R['mode'] == 'both')].id)
    missing = [i for i in PREREG_IDS if i not in ids]
    print(m, 'mode - missing IDs:', missing if missing else 'none')
print(R.groupby(['mode', 'id', 'status']).size().unstack(fill_value=0))
R.to_csv(RESULTS_PATH, index=False)
print('saved', RESULTS_PATH, len(R), 'rows')
R

clinical mode - missing IDs: none
natural mode - missing IDs: none
status       EXPLORATORY  FAIL  NOT_TESTABLE_ON_TL1.0  PASS  UNTESTABLE
mode     id                                                            
both     A1            0     0                      1     0           0
         B2            0     0                      1     0           0
         B9            0     0                      1     0           0
         C3            0     0                      1     0           0
         C4            0     0                      1     0           0
clinical A1            0     0                      2     0           0
         A2            1     0                      0     0           0
         A3            4     0                      0     0           0
         B1            0     3                      0     0           1
         B3            0     0                      0     3           1
         B4            0     0                      0     3          

,tl_version,prototype,mode,id,type,in_TL1_schedule,patient,lang,expected,pass_rule,...,estimate,ci_lo,ci_hi,n_twins,status,value_text,note,source_cell,notebook,run_time
0,1.0,Bravo,both,A1,confirmatory,False,all,all,Healthy twins near ceiling in dominant language on every battery section,>= 85% on every section,...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,"needs the test-battery sections, which only exist in TL2.0",1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
1,1.0,Bravo,both,B2,confirmatory,False,all,all,Passives harder than actives,"agrammatism ratio < 1, interval below 1",...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,TL1.0 has no passives (no non-canonical sentences),1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
2,1.0,Bravo,both,B9,exploratory,False,all,all,"English drops endings, Spanish swaps them",exploratory,...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,"TL1.0 verbs are whole words, so endings cannot be dropped or swapped",1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
3,1.0,Bravo,both,C3,confirmatory,False,all,all,Transfer larger for shared structures,interval of (shared minus specific) above 0,...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,TL1.0 was not designed with shared vs language-specific structures,1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
4,1.0,Bravo,both,C4,confirmatory,False,all,all,Maintenance reduces forgetting,interval of difference above 0,...,NaN,NaN,NaN,NaN,NOT_TESTABLE_ON_TL1.0,,no maintenance-therapy runs exist on TL1.0,1 Setup,S_Mission2_Step4_Baseline_112726,2026-11-27 21:59
5,1.0,Bravo,clinical,A1,confirmatory,False,P14,es,Healthy twins near ceiling in dominant language on every battery section,>= 85% on every section,...,98.90,98.2,99.6,3.0,NOT_TESTABLE_ON_TL1.0,"overall healthy strict per twin [99.6, 98.2, 98.9] (min-max shown as lo-hi; no battery sections)",information only,3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
6,1.0,Bravo,clinical,A1,confirmatory,False,P77,en,Healthy twins near ceiling in dominant language on every battery section,>= 85% on every section,...,95.47,94.6,96.1,3.0,NOT_TESTABLE_ON_TL1.0,"overall healthy strict per twin [94.6, 95.7, 96.1] (min-max shown as lo-hi; no battery sections)",information only,3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
7,1.0,Bravo,clinical,A2,confirmatory,False,both,en,Weaker language scales with exposure,more-exposed twin scores higher in L2,...,86.80,NaN,NaN,3.0,EXPLORATORY,P77 en 95.5 vs P14 en 8.7,would pass,3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
8,1.0,Bravo,clinical,A3,confirmatory,False,P14,en,Healthy twins rarely switch (single-language mode),<= 5% switches,...,0.00,0.0,0.0,3.0,EXPLORATORY,0.0% switches,would pass (<= 5%),3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27
9,1.0,Bravo,clinical,A3,confirmatory,False,P14,es,Healthy twins rarely switch (single-language mode),<= 5% switches,...,0.00,0.0,0.0,3.0,EXPLORATORY,0.0% switches,would pass (<= 5%),3 Checks,S_Mission2_Step4_Baseline_112726,2026-11-27 22:27


# 7. Summary — TL1.0 baseline (Step 4)

**Checks:** Section 1b printed identical: True, so the 5-sentence training file gives the same epoch-0 output (the first 44 runs used the full file). Healthy clinical-mode accuracy: P14 Spanish 98.9%, P77 English 95.5%. P14 English is untestable (about 9% healthy; only 7 English ditransitive test items).

**Preregistered results (clinical mode):** B3 PASS in all 3 testable cases (grammar errors rose 22–53 points more than word errors) and B4 PASS in all 3 (meaning damage raised word errors more), so the double dissociation holds. B1 FAIL in all 3: ditransitives were worst, but transitives scored higher than intransitives. B5 PASS for P77 (weak, interval crosses 0). B6 PASS for P77. B7 FAIL for P14 (48%), PASS for P77 (50.7%), which is chance level.

**Natural mode (exploratory):** B3 and B4 pattern again (double dissociation holds in both modes). Healthy P77 twins switch in 30% of English sentences, so clinical mode is the right main mode.

**Exploratory:** healthy twins switch 0% in clinical mode, but damaged twins switch often, so damage itself causes switching. Hidden transfer (C2, Bravo.2's Finding 2) is not supported once sham-adjusted intervals are used. One clear therapy effect: P77 Spanish therapy +50.3 [35.3, 65.3].

**What this means for TL2.0:** keep both lesion sites; test B1 more fairly (unergative intransitives, balanced frequencies, bigger balanced battery); transfer needs more twins; exact-match scoring needs preregistration v1.1 before TL2.0 testing.

**Decision-log rows:** clinical mode = main mode, natural mode reported separately; `_sl`/`_bl` run names; testability rule applied; dominant language from LUQ; 5-sentence training file for evaluations (verified identical).